# ARC26 Canon-CPT q9 with 24 inference prompts

Competition launcher matching the 31.94 Vanilla V2 configuration: q9 multi-token
DFS, threshold 0.2, eight geometries by three colour/order views, `score_kgmon`,
and an 11h50 wall-clock budget. The only model change is the premerged Canon-CPT
checkpoint plus jointly fine-tuned Canon-AC and rank-256 per-task LoRA during TTFT.


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "submit_competition"  # validation | submit_competition

from pathlib import Path

CODE_DATASET_ROOT = Path("/kaggle/input/datasets/yuvraj/arc2026")
PREMERGED_COMPETITION_ROOT = Path("/kaggle/input/arc26-canon-cpt-premerged-model/canon_cpt_premerged")
PREMERGED_NOTEBOOK_ROOT = Path("/kaggle/input/notebooks/yuvraj/arc26-canon-cpt-premerged-model/canon_cpt_premerged")
COMP_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
MODERN_UTILITY_ROOT = Path("/kaggle/usr/lib/notebooks/yuvraj/pip_install_unsloth_ddp_repair")
FA2_ROOT = Path("/kaggle/input/notebooks/yuvraj/flash-attention-cu13-torch-2-11-cp312/flash_attn_cu13_torch211_cp312")

VALIDATION_KEYS = None
NPROCS = 4
DFS_PROB_THRESHOLD = 0.2
UNSLOTH_MULTITOKEN_REPEAT_LEN = 9
EVAL_COLOR_PERMUTATIONS = 3
SELECTION_ALGORITHM = "score_kgmon"
PROFILE_TIMINGS = True

VALIDATION_END_TIME_HOURS = 2.5
SUBMIT_COMPETITION_END_TIME_HOURS = 11 + 50 / 60
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = Path("/kaggle/working/arc26_canon_cpt_submit")
WORK_CODE_DIR = WORK_NOTEBOOK_ROOT / "ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = Path("/kaggle/working/canon_q9_submit_stack")


In [ ]:
import os


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    return value is not None and value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE
PREMERGED_ROOT = PREMERGED_COMPETITION_ROOT if IS_KAGGLE_RERUN else PREMERGED_NOTEBOOK_ROOT
MODEL_PATH = PREMERGED_ROOT / "model"
CANON_STATE = PREMERGED_ROOT / "canon_ac.pt"

EVAL_CHALLENGES = COMP_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = COMP_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = COMP_ROOT / "arc-agi_test_challenges.json"

if IS_KAGGLE_RERUN:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = Path("/kaggle/working/canon_cpt_q9_24_submit_candidates")
    SUBMISSION_PATH = Path("/kaggle/working/submission.json")
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS
    RUN_INFERENCE = True
elif MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR = Path("/kaggle/working/canon_cpt_q9_24_validation_candidates")
    SUBMISSION_PATH = Path("/kaggle/working/canon_cpt_q9_24_validation_submission.json")
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
    RUN_INFERENCE = True
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = Path("/kaggle/working/canon_cpt_q9_24_shortcut")
    SUBMISSION_PATH = Path("/kaggle/working/submission.json")
    SELECTED_KEYS = None
    END_TIME_HOURS = 0.0
    RUN_INFERENCE = False

print("mode_requested =", MODE)
print("is_kaggle_rerun =", IS_KAGGLE_RERUN)
print("test_path =", TEST_PATH)
print("end_time_hours =", END_TIME_HOURS)
print("run_inference =", RUN_INFERENCE)


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

required_files = [
    CODE_DATASET_ROOT / "ARC-AGI1/qwen_baseline/starter.py",
    TEST_PATH,
    MODERN_UTILITY_ROOT / "unsloth/__init__.py",
    FA2_ROOT / "flash_attn/__init__.py",
]
if RUN_INFERENCE:
    required_files.extend([MODEL_PATH / "config.json", CANON_STATE])
for required in required_files:
    if not required.is_file():
        raise FileNotFoundError(required)

for path in (WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT):
    shutil.rmtree(path, ignore_errors=True)
if RESET_RUN_ARTIFACTS:
    SUBMISSION_PATH.unlink(missing_ok=True)
shutil.copytree(CODE_DATASET_ROOT, WORK_NOTEBOOK_ROOT)

environment = os.environ.copy()
old_parts = [
    part for part in environment.get("PYTHONPATH", "").split(os.pathsep)
    if part and "pip_install_unsloth_" not in part and "flash_attention_" not in part
]
environment.update({
    "PYTHONPATH": os.pathsep.join([
        str(FA2_ROOT), str(WRITABLE_UNSLOTH_PARENT), str(MODERN_UTILITY_ROOT),
        "/kaggle/working", str(WORK_CODE_DIR), *old_parts,
    ]),
    "PYTHONPYCACHEPREFIX": "/kaggle/working/python_cache",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "3",
    "PYTHONUNBUFFERED": "1",
})
print("model =", MODEL_PATH)
print("canon state =", CANON_STATE)


In [ ]:
writable_unsloth = WRITABLE_UNSLOTH_PARENT / "unsloth"
shutil.copytree(MODERN_UTILITY_ROOT / "unsloth", writable_unsloth)
subprocess.run(
    [
        sys.executable,
        str(WORK_CODE_DIR / "patch_unsloth_qwen3_multitoken.py"),
        "--unsloth-package-dir", str(writable_unsloth),
    ],
    env=environment,
    check=True,
)
print("patched writable Unsloth =", writable_unsloth)


In [ ]:
# Source-only production preflight; no model load on save-version runs.
starter_source = (WORK_CODE_DIR / "starter.py").read_text()
solver_source = (WORK_CODE_DIR / "arc_solver.py").read_text()
search_source = (WORK_CODE_DIR / "arc_search_multitoken.py").read_text()
for marker in (
    "--canon-ac-state",
    "--use-unsloth-multitoken-dfs",
    "--eval-color-permutations",
):
    if marker not in starter_source:
        raise RuntimeError(f"Missing starter marker: {marker}")
if "Canon TTFT delta_l2=" not in solver_source:
    raise RuntimeError("arc2026 lacks joint Canon TTFT")
if "del outputs" not in search_source:
    raise RuntimeError("arc2026 lacks q9 recursive-output release fix")
print("Canon competition preflight passed")


In [ ]:
import json
import time

if RUN_INFERENCE:
    command = [
        sys.executable, str(WORK_CODE_DIR / "starter.py"),
        "--test-path", str(TEST_PATH),
        "--model-path", str(MODEL_PATH),
        "--output-dir", str(OUTPUT_DIR),
        "--nprocs", str(NPROCS),
        "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
        "--eval-color-permutations", str(EVAL_COLOR_PERMUTATIONS),
        "--ttft-method", "full_sft",
        "--canon-ac-state", str(CANON_STATE),
        "--use-unsloth-multitoken-dfs",
        "--unsloth-multitoken-repeat-len", str(UNSLOTH_MULTITOKEN_REPEAT_LEN),
        "--end-time", str(time.time() + END_TIME_HOURS * 3600),
    ]
    if PROFILE_TIMINGS:
        command.append("--profile-timings")
    if SELECTED_KEYS is not None:
        command.extend(["--keys-json", json.dumps(SELECTED_KEYS)])
    print("running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=WORK_CODE_DIR, env=environment, check=True)
else:
    print("save-version shortcut: full inference runs only during competition rerun")


In [ ]:
import json
import os
import sys
from pathlib import Path

WORK_CODE_DIR = str(WORK_CODE_DIR)
if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon, score_full_probmul_3

SELECTION_ALGORITHMS = {
    "score_kgmon": score_kgmon,
    "score_full_probmul_3": score_full_probmul_3,
}


def _decoded_basekeys(output_dir):
    p = Path(output_dir)
    if not p.exists():
        return []
    return sorted({x.name.split(".")[0] for x in p.iterdir() if x.is_file()})


data = ArcDataset.from_file(TEST_PATH)
if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    data = data.load_replies(SOLUTION_PATH)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    decoder.load_decoded_results(OUTPUT_DIR)

selection_algorithm = SELECTION_ALGORITHMS[SELECTION_ALGORITHM]
submission = data.get_submission(decoder.run_selection_algo(selection_algorithm) if decoder.decoded_results else None)

with open(str(SUBMISSION_PATH), "w") as f:
    json.dump(submission, f)

print("decoded_output_keys =", len(decoder.decoded_results))
print("decoded_basekeys =", _decoded_basekeys(OUTPUT_DIR)[:20])
print("submission_path =", SUBMISSION_PATH)
print("submission_tasks =", len(submission))
print("submission_exists =", Path(SUBMISSION_PATH).exists())

if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    if decoder.decoded_results:
        decoder.benchmark_selection_algos()
    with open(str(SUBMISSION_PATH)) as f:
        reload_submission = json.load(f)
    print("validation_score =", data.validate_submission(reload_submission))
else:
    preview_keys = list(submission)[:5]
    print("preview_keys =", preview_keys)
